# Music Auto-tagging

In [3]:
import torch
import torchaudio
import IPython.display as ipd
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
!pip install --upgrade gdown
!gdown 15e9E3oZdudErkPKwb0rCAiZXkPxdZkV6
# !wget https://sogang365-my.sharepoint.com/:u:/g/personal/dasaem_jeong_o365_sogang_ac_kr/EdkHWV-qvxBEi-d0Ua73VG4BEp7EZO7HMvrXsWqeJvMJzg?e=GbYylV&download=1

!unzip -q mtat_8000.zip

Downloading...
From (original): https://drive.google.com/uc?id=15e9E3oZdudErkPKwb0rCAiZXkPxdZkV6
From (redirected): https://drive.google.com/uc?id=15e9E3oZdudErkPKwb0rCAiZXkPxdZkV6&confirm=t&uuid=7cd9d39d-a9ba-4778-bb1f-38a026cb7668
To: /content/mtat_8000.zip
100% 921M/921M [00:13<00:00, 66.9MB/s]


In [10]:
# check dataset
data_dir = Path('MTAT_SMALL')
assert data_dir.exists()

mp3_fns = list(data_dir.rglob('*.mp3'))
len(mp3_fns)

mp3_fn = mp3_fns[0]
y, sr = torchaudio.load(mp3_fn)
print(mp3_fn, sr)
ipd.Audio(y, rate=sr)

MTAT_SMALL/8/justin_bianco-phoenix-12-framework-0-29.mp3 16000


In [23]:
df = pd.read_csv('MTAT_SMALL/meta.csv', index_col=[0])
df.columns.values[1:-1]

array(['singer', 'harpsichord', 'sitar', 'heavy', 'foreign', 'no piano',
       'classical', 'female', 'jazz', 'guitar', 'quiet', 'solo', 'folk',
       'ambient', 'new age', 'synth', 'drum', 'bass', 'loud', 'string',
       'opera', 'fast', 'country', 'violin', 'electro', 'trance', 'chant',
       'strange', 'modern', 'hard', 'harp', 'pop', 'female vocal',
       'piano', 'orchestra', 'eastern', 'slow', 'male', 'vocal',
       'no singer', 'india', 'rock', 'dance', 'cello', 'techno', 'flute',
       'beat', 'soft', 'choir', 'baroque'], dtype=object)

In [17]:
str(mp3_fn.relative_to('MTAT_SMALL/'))

'8/justin_bianco-phoenix-12-framework-0-29.mp3'

In [19]:
df[df['mp3_path']==str(mp3_fn.relative_to('MTAT_SMALL/'))]

,Unnamed: 0,clip_id,singer,harpsichord,sitar,heavy,foreign,no piano,classical,female,...,rock,dance,cello,techno,flute,beat,soft,choir,baroque,mp3_path
4614,21858,48021,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,8/justin_bianco-phoenix-12-framework-0-29.mp3


In [ ]:
from tqdm.auto import tqdm

class MTATDataset:
  def __init__(self, dir_path, split='train', num_max_data=6000, sr=16000):
    self.dir = Path(dir_path)
    self.labels = pd.read_csv(self.dir / "meta.csv", index_col=[0])
    self.sr = sr

    if split=="train":
      sub_dir_ids = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c']
    elif split=='valid':
      sub_dir_ids = ['d']
    elif split=='test': #test
      sub_dir_ids = ['e', 'f', 'g']
    else:
      raise NotImplementedError

    is_in_set = [True if x[0] in sub_dir_ids else False for x in self.labels['mp3_path'].values.astype('str')]
    self.labels = self.labels.iloc[is_in_set] # filter label by is_in_set
    self.labels = self.labels[:num_max_data]
    self.vocab = self.labels.columns.values[1:-1]
    self.label_tensor = self.convert_label_to_tensor()
    self.audios = self.load_audio()

  def convert_label_to_tensor(self):
    return torch.tensor(self.labels.values[:, 1:-1].astype('bool'), dtype=torch.float)

  def load_audio(self):
    audios = []
    for idx in tqdm(range(len(self))):
      info = self.labels.iloc[idx]
      mp3_path = self.dir / info['mp3_path']
      audio, sr = torchaudio.load(mp3_path)
      audios.append(audio)
    return audios


  def __len__(self):
    return len(self.labels)

  def __getitem__(self, idx):
    # info = self.labels.iloc[idx]
    # mp3_path = self.dir / info['mp3_path']

    # audio, sr = torchaudio.load(mp3_path)
    # assert sr == self.sr
    audio = self.audios[idx]
    label = self.label_tensor[idx]
    return audio, label

train_set = MTATDataset('MTAT_SMALL')

audio, label = train_set[4000]
ipd.display(ipd.Audio(audio, rate=train_set.sr, normalize=False))
print(label)

# convert multi-hot label to readable tag
actviated_tag_idxs = torch.where(label)[0]
train_set.vocab[actviated_tag_idxs]

  0%|          | 0/5000 [00:00<?, ?it/s]

In [77]:
import torch.nn as nn
class AutoTagger(nn.Module):
  def __init__(self, out_size=50):
    super().__init__()
    self.mel = torchaudio.transforms.MelSpectrogram(n_fft=2048, hop_length=1024, n_mels=80)
    self.db = torchaudio.transforms.AmplitudeToDB()
    self.conv_stack = nn.Sequential(
        nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3),
        nn.ReLU(),
        nn.MaxPool2d(2), # kernel_size can be in tuple.
        nn.Conv2d(16, 32, 3),
        nn.ReLU(),
        nn.MaxPool2d(2), # kernel_size can be in tuple.
        nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3),
        nn.ReLU(),
        nn.MaxPool2d(2) # kernel_size can be in tuple.
    )
    self.final_pool = nn.AdaptiveMaxPool1d(1) # fix output size, rather than kernel size
    self.proj = nn.Linear(512, out_size) # 512 only works for specific height of input

  def forward(self, x):
    # print(f"Audio shape: {x.shape}")
    x = self.mel(x)
    x = self.db(x) / 80
    # print(f"Mel shape: {x.shape}")
    x = self.conv_stack(x)
    # print(f"Conv out shape: {x.shape}")

    x = x.flatten(-3, -2) # N x C x H x W -> N x (CxH) x W
    # print(f"flatten shape: {x.shape}")

    # x = torch.max(x, dim=-1)[0] # take max pool over time axis. and use [0] for getting value. [1] is index
    x = self.final_pool(x).squeeze(-1)
    x = self.proj(x)


    return x

model = AutoTagger()
out = model(audio)
out.shape

torch.Size([32, 50])

In [55]:
cnn_layer = nn.Conv2d(in_channels=1,
                      out_channels=5,
                      kernel_size=3,
                      stride=2, # stride can replace max pool
                      padding=1,
                      dilation=1)

print(spec.shape)
# cnn layer expects input to be N x C x H x W (or C x H x W)
unsq_spec = spec.unsqueeze(0)
print(unsq_spec.shape)
cnn_out = cnn_layer(unsq_spec)
print(f"cnn_out.shape is {cnn_out.shape}")

torch.Size([80, 456])
torch.Size([1, 80, 456])
cnn_out.shape is torch.Size([5, 40, 228])


In [78]:
train_loader = torch.utils.data.DataLoader(train_set, batch_size=32, shuffle=True)

batch = next(iter(train_loader))
audio, label = batch

out = model(audio)
out.shape

torch.Size([32, 50])

In [82]:
prob = torch.sigmoid(out)
prob

tensor([[0.4884, 0.4970, 0.5030,  ..., 0.4918, 0.5258, 0.5038],
        [0.4886, 0.4969, 0.5022,  ..., 0.4921, 0.5252, 0.5038],
        [0.4853, 0.4962, 0.5011,  ..., 0.4938, 0.5257, 0.5047],
        ...,
        [0.4880, 0.4971, 0.5028,  ..., 0.4921, 0.5256, 0.5044],
        [0.4858, 0.4987, 0.5049,  ..., 0.4949, 0.5273, 0.5048],
        [0.4817, 0.4958, 0.5027,  ..., 0.4943, 0.5277, 0.5027]],
       grad_fn=<SigmoidBackward0>)

In [89]:
def get_binary_cross_entropy(pred, target):
  loss = -(target * torch.log(pred) + (1-target) * torch.log(1-pred))
  return loss.mean()

# - (y log(y_hat) +  (1-y)log(1-y_hat) )
loss = get_binary_cross_entropy(prob, label)
loss

tensor(0.6959, grad_fn=<MeanBackward0>)

In [86]:
print(prob.shape, label.shape)

torch.Size([32, 50]) torch.Size([32, 50])
